# permute-back-argsort — worked example 1: Why argsort inverts a permutation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `permute-back-argsort`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Given a permutation `dims = (2, 0, 1)`, `numpy.argsort(dims)` returns the inverse permutation — the array telling you where each axis went. Specifically, `argsort(dims)[i]` is the position at which value `i` appears in `dims`. For `(2, 0, 1)`: value 0 is at position 1, value 1 is at position 2, value 2 is at position 0, so `argsort((2,0,1)) = (1, 2, 0)`. Permuting by this inverse undoes the original permutation.

## Worked solution

**Step 1 — State the problem.** Forward: `y = x.permute(2, 0, 1)` maps axis 0→2, 1→0, 2→1 in `y`. To backpropagate `grad_out` (same shape as `y`) back to `x`, we need to undo that reordering.

**Step 2 — Compute the inverse with argsort.** `np.argsort([2, 0, 1])` asks: where is 0 in `[2,0,1]`? At index 1. Where is 1? At index 2. Where is 2? At index 0. Result: `[1, 2, 0]`.

**Step 3 — Apply the inverse permutation.** `grad_out.permute(1, 2, 0)` moves `grad_out`'s axis 0 back to where axis 0 was in `x`. After this, the gradient tensor has the same axis order as `x`.

**Step 4 — Verify the round-trip.** We check that `x.permute(2,0,1).permute(1,2,0)` equals `x` exactly (bit-for-bit, since no arithmetic is involved). The round-trip identity always holds for any valid permutation.

In [ ]:
import torch as t
import numpy as np

def permute_back_explained(grad_out: t.Tensor, x: t.Tensor, dims: tuple) -> t.Tensor:
    """Backward of permute: apply inverse permutation via argsort."""
    inverse = tuple(int(i) for i in np.argsort(dims))
    return grad_out.permute(*inverse)

# Demonstrate with dims=(2,0,1) on a (2,3,4) tensor
t.manual_seed(22)
x = t.randn(2, 3, 4)
dims = (2, 0, 1)

y = x.permute(*dims)         # shape (4, 2, 3)
print(f'x shape: {x.shape}')      # (2, 3, 4)
print(f'y shape: {y.shape}')      # (4, 2, 3)

inverse_dims = tuple(int(i) for i in np.argsort(dims))
print(f'Original dims: {dims}')       # (2, 0, 1)
print(f'Inverse dims:  {inverse_dims}')  # (1, 2, 0)

grad_out = t.ones_like(y)                  # same shape as y
grad_x = permute_back_explained(grad_out, x, dims)
print(f'grad_x shape: {grad_x.shape}')    # (2, 3, 4) -- matches x

# Round-trip identity: y.permute(inverse) == x
print(f'Round-trip exact: {t.equal(y.permute(*inverse_dims), x)}')  # True